In [6]:
from brian2 import *
import pandas as pd
import os, time

prefs.codegen.target = 'numpy'
file_path = "database_gEIN.csv"
# ============================================================
# CONFIGURATION
# ============================================================
PASS_LETTER = "A"
TRIALS_TO_RUN = [1, 2]
START_SEED = 1
sweep_parameter = "gEIN"          
sweep_values = [0.130, 0.12675]

# Load or create dataset
if os.path.exists(file_path):
    df = pd.read_csv(file_path)
else:
    df = pd.DataFrame()

# ============================================================
# MAIN SWEEP LOOP
# ============================================================

for value in sweep_values:
    print(f"\n\n==============================")
    print(f" RUNNING {sweep_parameter} = {value}")
    print(f"==============================\n")

    for trial in TRIALS_TO_RUN:
        print(f"\n=== TRIAL {trial} ===")

        # Reset Brian2
        start_scope()

        # Create seed label like A01, A02, ..., A10
        seed_label = f"{PASS_LETTER}{trial:02d}"

        # Numeric seed for RNG (1–10)
        numeric_seed = trial

        # Apply randomness
        np.random.seed(numeric_seed)
        seed(numeric_seed)
        import random
        random.seed(numeric_seed)

        # ============================================================
        # PARAMETER OVERRIDES
        # ============================================================

        # Default values
        coh = 15
        gEEN = 0.165 * nS / 1600 * 1600
        gEIN = 0.130 * nS / 1600 * 1600

        # Override the chosen parameter
        if sweep_parameter == "coh":
            coh = value

        elif sweep_parameter == "gEEN":
            gEEN = value * nS / 1600 * 1600

        elif sweep_parameter == "gEIN":
            gEIN = value * nS / 1600 * 1600

        elif sweep_parameter == "gIE":
            gIE = value * nS / 400 * 400

        # ============================================================
        # YOUR FULL MODEL (unchanged)
        # ============================================================

        # Stimulus and simulation parameters
        sigma = 4.0 * Hz
        mu0 = 40.0 * Hz
        mu1 = 40.0 * Hz
        stim_interval = 50.0 * ms
        stim_on = 1000 * ms
        stim_off = 3000 * ms
        runtime = 4000 * ms

        N_ext = 1000
        rate_ext_E = 2400 * Hz / N_ext
        rate_ext_I = 2400 * Hz / N_ext

        N = 2000
        f_inh = 0.2
        NE = int(N * (1 - f_inh))
        NI = int(N * f_inh)
        fE = 0.15
        subN = int(fE * NE)

        El = -70*mV
        Vt = -50*mV
        Vr = -55*mV
        CmE = 0.5*nF
        CmI = 0.2*nF
        gLeakE = 25*nS
        gLeakI = 20*nS
        refE = 2*ms
        refI = 1*ms

        V_E = 0*mV
        V_I = -70*mV
        tau_AMPA = 2*ms
        tau_NMDA_rise = 2*ms
        tau_NMDA_decay = 100*ms
        tau_GABA = 5*ms
        alpha = 0.5*kHz
        C = 1*mmole

        gextE = 2.1*nS
        gextI = 1.62*nS
        gEEA = 0.05*nS / NE * 1600
        gEIA = 0.04*nS / NE * 1600
        # gEEN already overridden above
        # gEIN already overridden above
        gIE  = 1.3   * nS / 400  * 400
        gII = 1.0*nS / NI * 400

        Jp = 1.7
        Jm = 1.0 - fE*(Jp-1)/(1-fE)

        eqsE = """
           label : integer (constant)
           dV/dt = (- gLeakE*(V-El) - I_AMPA - I_NMDA - I_GABA - I_AMPA_ext + I_input)/CmE : volt (unless refractory)
           I_AMPA = s_AMPA*(V-V_E) : amp
           ds_AMPA/dt = -s_AMPA/tau_AMPA : siemens
           I_NMDA = gEEN*s_NMDA_tot*(V-V_E)/(1+exp(-0.062*V/mvolt)*(C/mmole/3.57)) : amp
           s_NMDA_tot : 1
           I_GABA = s_GABA*(V-V_I) : amp
           ds_GABA/dt = -s_GABA/tau_GABA : siemens
           I_AMPA_ext = s_AMPA_ext*(V-V_E) : amp
           ds_AMPA_ext/dt = -s_AMPA_ext/tau_AMPA : siemens
           I_input : amp
           ds_NMDA/dt = -s_NMDA/tau_NMDA_decay + alpha*x*(1-s_NMDA) : 1
           dx/dt = -x/tau_NMDA_rise : 1
        """

        eqsI = """
           dV/dt = (- gLeakI*(V-El) - I_AMPA - I_NMDA - I_GABA - I_AMPA_ext)/CmI : volt (unless refractory)
           I_AMPA = s_AMPA*(V-V_E) : amp
           ds_AMPA/dt = -s_AMPA/tau_AMPA : siemens
           I_NMDA = gEIN*s_NMDA_tot*(V-V_E)/(1+exp(-0.062*V/mvolt)*(C/mmole/3.57)) : amp
           s_NMDA_tot : 1
           I_GABA = s_GABA*(V-V_I) : amp
           ds_GABA/dt = -s_GABA/tau_GABA : siemens
           I_AMPA_ext = s_AMPA_ext*(V-V_E) : amp
           ds_AMPA_ext/dt = -s_AMPA_ext/tau_AMPA : siemens
        """

        popE = NeuronGroup(NE, eqsE, threshold='V>Vt', reset='V=Vr', refractory=refE, method='euler')
        popI = NeuronGroup(NI, eqsI, threshold='V>Vt', reset='V=Vr', refractory=refI, method='euler')

        popE1 = popE[:subN]
        popE2 = popE[subN:2*subN]
        popE3 = popE[2*subN:]
        popE1.label = 0
        popE2.label = 1
        popE3.label = 2

        C_EE_AMPA = Synapses(popE, popE, 'w:siemens', on_pre='s_AMPA += w', delay=0.5*ms)
        C_EE_AMPA.connect()
        C_EE_AMPA.w[:] = gEEA
        C_EE_AMPA.w["label_pre==label_post and label_pre<2"] = gEEA*Jp
        C_EE_AMPA.w["label_pre!=label_post and label_post<2"] = gEEA*Jm

        C_EI_AMPA = Synapses(popE, popI, on_pre='s_AMPA += gEIA', delay=0.5*ms)
        C_EI_AMPA.connect()

        C_EE_NMDA = Synapses(popE, popE, on_pre='x_pre += 1', delay=0.5*ms)
        C_EE_NMDA.connect(j='i')

        NMDA_sum_group = NeuronGroup(3, 's:1')
        NMDA_sum = Synapses(popE, NMDA_sum_group, 's_post = s_NMDA_pre : 1 (summed)')
        NMDA_sum.connect(j='label_pre')

        NMDA_set_total_E = Synapses(NMDA_sum_group, popE,
            '''w:1
               s_NMDA_tot_post = w*s_pre : 1 (summed)''')
        NMDA_set_total_E.connect()
        NMDA_set_total_E.w = 1
        NMDA_set_total_E.w["i==label_post and label_post<2"] = Jp
        NMDA_set_total_E.w["i!=label_post and label_post<2"] = Jm

        NMDA_set_total_I = Synapses(NMDA_sum_group, popI,
            's_NMDA_tot_post = s_pre : 1 (summed)')
        NMDA_set_total_I.connect()

        C_IE = Synapses(popI, popE, on_pre='s_GABA += gIE', delay=0.5*ms)
        C_IE.connect()

        C_II = Synapses(popI, popI, on_pre='s_GABA += gII', delay=0.5*ms)
        C_II.connect()

        extinputE = PoissonInput(popE, 's_AMPA_ext', N_ext, rate_ext_E, gextE)
        extinputI = PoissonInput(popI, 's_AMPA_ext', N_ext, rate_ext_I, gextI)

        stiminputE1 = PoissonGroup(subN, rates=0*Hz)
        stiminputE2 = PoissonGroup(subN, rates=0*Hz)
        stiminputE1.run_regularly("rates = int(t>stim_on and t<stim_off)*(mu0 + coh/100*mu1 + sigma*randn())", dt=stim_interval)
        stiminputE2.run_regularly("rates = int(t>stim_on and t<stim_off)*(mu0 - coh/100*mu1 + sigma*randn())", dt=stim_interval)

        C_stimE1 = Synapses(stiminputE1, popE1, on_pre='s_AMPA_ext += gextE')
        C_stimE1.connect(j='i')
        C_stimE2 = Synapses(stiminputE2, popE2, on_pre='s_AMPA_ext += gextE')
        C_stimE2.connect(j='i')

        popE.s_NMDA_tot = tau_NMDA_decay * 10*Hz * 0.2
        popI.s_NMDA_tot = tau_NMDA_decay * 10*Hz * 0.2
        popE.V = Vt - 2*mV
        popI.V = Vt - 2*mV

        SME1 = SpikeMonitor(popE1)
        SME2 = SpikeMonitor(popE2)
        R1 = PopulationRateMonitor(popE1)
        R2 = PopulationRateMonitor(popE2)

        # ============================================================
        # RUN TRIAL
        # ============================================================
        run(runtime, report='stdout')

        # ============================================================
        # METRICS
        # ============================================================
        rate1 = R1.smooth_rate(window='gaussian', width=50*ms)
        rate2 = R2.smooth_rate(window='gaussian', width=50*ms)

        winner = 1 if max(rate1) > max(rate2) else 0

        threshold = 10*Hz
        cross1 = np.where(rate1 > threshold)[0]
        cross2 = np.where(rate2 > threshold)[0]

        dt1 = R1.t[cross1[0]]/ms if len(cross1)>0 else None
        dt2 = R2.t[cross2[0]]/ms if len(cross2)>0 else None
        decision_time = dt1 if winner==1 else dt2

        peak1 = np.max(rate1)/Hz
        peak2 = np.max(rate2)/Hz

        # ============================================================
        # SAVE ROW
        # ============================================================
        
        noise_E = float(rate_ext_E / Hz)
        noise_I = float(rate_ext_I / Hz)
        
        new_row = {
            "coh": float(coh),
            "gEIN": float(gEIN / nS),
            #"gEEN": float(gEEN / nS),
            #"gIE":  float(gIE  / nS),
            "acc": int(winner),
            "dt1": float(dt1) if dt1 is not None else None,
            "dt2": float(dt2) if dt2 is not None else None,
            "dt":  float(decision_time) if decision_time is not None else None,
            "peak1": float(peak1),
            "peak2": float(peak2),
            "noise_E": noise_E,
            "noise_I": noise_I,
            "seed": seed_label
        }



        df = pd.concat([df, pd.DataFrame([new_row])], ignore_index=True)

        # ============================================================
        # SAFE SAVE
        # ============================================================
        saved = False
        while not saved:
            try:
                df.to_csv(file_path, index=False)
                saved = True
            except PermissionError:
                print("Excel is open — close it so I can save.")
                time.sleep(2)

print("\n=== ALL SWEEP TRIALS COMPLETE ===")




 RUNNING gEIN = 0.13


=== TRIAL 1 ===
Starting simulation at t=0. s for a duration of 4. s
0.5479 s (13%) simulated in 10s, estimated 1m 3s remaining.
1.0837 s (27%) simulated in 20s, estimated 54s remaining.
1.414 s (35%) simulated in 30s, estimated 55s remaining.
1.5832 s (39%) simulated in 40s, estimated 1m 1s remaining.
1.9005 s (47%) simulated in 50s, estimated 55s remaining.
2.5419 s (63%) simulated in 1m 0s, estimated 34s remaining.
3.0801 s (77%) simulated in 1m 10s, estimated 21s remaining.
3.3734 s (84%) simulated in 1m 20s, estimated 15s remaining.
3.706 s (92%) simulated in 1m 30s, estimated 7s remaining.
4. s (100%) simulated in 1m 34s

=== TRIAL 2 ===
Starting simulation at t=0. s for a duration of 4. s
0.392 s (9%) simulated in 10s, estimated 1m 32s remaining.
0.7379 s (18%) simulated in 20s, estimated 1m 28s remaining.
1.0173 s (25%) simulated in 30s, estimated 1m 28s remaining.
1.3305 s (33%) simulated in 40s, estimated 1m 20s remaining.
1.8348 s (45%) simulated in 